# TurboQuant cuTile Kernels — Blackwell B200

End-to-end demo: cuTile kernels for TurboQuant asymmetric attention on a Blackwell B200 GPU.
Covers kernel correctness, latency benchmarks, roofline analysis, and transformer model integration.

Paper: *TurboQuant: Online Vector Quantization with Near-optimal Distortion Rate* (ICLR 2026)

## 0. Install packages (run once, then restart kernel)

In [ ]:
import sys
import subprocess

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=True)

subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)
pip("torch", "--index-url", "https://download.pytorch.org/whl/cu128")
pip("cuda-tile[tileiras]", "--extra-index-url", "https://pypi.nvidia.com")
pip("scipy", "matplotlib", "numpy")
pip("transformers", "accelerate", "sentencepiece")

print("\nDone. RESTART YOUR KERNEL before running the cells below.")

## 1. Environment check

In [ ]:
import torch
import cuda.tile as ct
import math
import time
import sys

assert torch.cuda.is_available(), "No CUDA- this notebook needs a GPU(Blackwell or Ampere)"

print(f"PyTorch  {torch.__version__}")
print(f"CUDA     {torch.version.cuda}")
print(f"cuTile   {ct.__version__ if hasattr(ct, '__version__') else 'loaded'}")
print(f"GPU      {torch.cuda.get_device_name()}")

In [ ]:
sys.path.insert(0, ".")
from turboquant_cutile import TurboQuantEngine, LloydMaxCodebook
from turboquant_cutile.constants import BLOCK_Q, BLOCK_KV, BLOCK_S, HEAD_DIM

from turboquant_cutile.compress import (
    turboquant_compress_2bit, turboquant_compress_3bit,
    turboquant_compress_values_3bit, turboquant_compress_values_2bit,
)
from turboquant_cutile.attention import turboquant_attention_scores, turboquant_fused_attention
from turboquant_cutile.decompress import turboquant_decompress_3bit, turboquant_decompress_2bit

print("All cuTile kernel modules imported successfully")

## 2. Setup

In [ ]:
DEVICE = "cuda"
TOTAL_BITS = 3   # 2-bit MSE + 1-bit QJL
SEED = 42

engine = TurboQuantEngine(
    head_dim=HEAD_DIM,
    total_bits=TOTAL_BITS,
    seed=SEED,
    device=DEVICE,
)

print(f"device={DEVICE}, total_bits={TOTAL_BITS}, mse_bits={engine.mse_bits}")
print(f"Pi: {engine.Pi.shape} on {engine.Pi.device}")
print(f"S:  {engine.S.shape} on {engine.S.device}")
print(f"Key codebook ({engine.mse_bits}-bit): {engine.key_codebook.n_levels} levels")
print(f"Val codebook ({TOTAL_BITS}-bit): {engine.val_codebook.n_levels} levels")

### Generate synthetic Q vector and KV cache

In [ ]:
SEQ_K = 2048
torch.manual_seed(SEED)

K = torch.randn(SEQ_K, HEAD_DIM, device=DEVICE, dtype=torch.float16)
V = torch.randn(SEQ_K, HEAD_DIM, device=DEVICE, dtype=torch.float16)
Q = torch.randn(1, HEAD_DIM, device=DEVICE, dtype=torch.float16)

print(f"K: {K.shape}  V: {V.shape}  Q: {Q.shape}")

## 3. cuTile key compression kernel

In [ ]:
# launch cuTile compression kernel
compressed_k = engine.launch_compress_keys(K)

print("cuTile key compression output:")
for name, val in compressed_k.items():
    if isinstance(val, torch.Tensor):
        print(f"  {name:>16s}: {str(val.shape):>20s}  {val.dtype}")

# sanity checks
assert compressed_k["indices"].max() < engine.key_codebook.n_levels
assert set(compressed_k["qjl_signs"].unique().tolist()) == {-1, 1}
assert (compressed_k["vec_norms"] > 0).all()

# measure reconstruction quality with cosine similarity
k_cos = torch.nn.functional.cosine_similarity(
    K.float(), compressed_k["k_mse"].float(), dim=-1
).mean()

print(f"\nKey reconstruction cosine sim: {k_cos:.4f}")

## 4. cuTile value compression + decompression

In [ ]:
# cuTile compression (V matrix)
compressed_v = engine.launch_compress_values(V)

print("cuTile value compression output:")
for name, val in compressed_v.items():
    if isinstance(val, torch.Tensor):
        print(f"  {name:>16s}: {str(val.shape):>20s}  {val.dtype}")

# cuTile decompression
V_recon = engine.launch_decompress_values(compressed_v)

assert V_recon.shape == V.shape
assert compressed_v["indices"].max() < engine.val_codebook.n_levels

# measure reconstruction quality with cosine similarity
v_cos = torch.nn.functional.cosine_similarity(
    V.float(), V_recon.float(), dim=-1
).mean()
print(f"\nValue reconstruction cosine sim: {v_cos:.4f}")
print(f"V_recon: {V_recon.shape} {V_recon.dtype}")

## 5. cuTile attention scores kernel

In [ ]:
# ground truth: standard FP16 matmul
true_scores = (Q.float() @ K.float().T) * engine.scale

# cuTile basic attention scores kernel (no fused softmax or values matmul)
tq_scores = engine.launch_attention_scores(Q, compressed_k)

assert tq_scores.shape == (1, SEQ_K)

corr = torch.corrcoef(torch.stack([true_scores.flatten(), tq_scores.flatten()]))[0, 1]
score_mse = ((true_scores - tq_scores) ** 2).mean()

print(f"Score correlation:  {corr:.4f}")
print(f"Score MSE:          {score_mse:.6f}")
print(f"True top-1:         {true_scores.argmax().item()}")
print(f"TQ top-1:           {tq_scores.argmax().item()}")

## 6. cuTile fused attention kernel (full pipeline)

In [ ]:
# cuTile: compress -> attend -> softmax -> weighted V sum
tq_output = engine.launch_fused_attention(Q, compressed_k, compressed_v)

# FP16 reference
ref_weights = torch.softmax(true_scores, dim=-1)
ref_output = (ref_weights @ V.float()).half()

output_cos = torch.nn.functional.cosine_similarity(
    ref_output.float().flatten().unsqueeze(0),
    tq_output.float().flatten().unsqueeze(0),
).item()

print(f"Output cosine similarity vs FP16: {output_cos:.4f}")
print(f"Output shape: {tq_output.shape}  dtype: {tq_output.dtype}")

## 7. Swizzle correctness check

In [ ]:
# baseline (no swizzle) vs optimized (swizzle + exp2 + approx div + pipelined loads)
tq_scores_swz = engine.launch_attention_scores(Q, compressed_k, use_swizzle=True)
tq_output_swz = engine.launch_fused_attention(Q, compressed_k, compressed_v, use_swizzle=True)

score_cos = torch.nn.functional.cosine_similarity(
    tq_scores.float().flatten().unsqueeze(0),
    tq_scores_swz.float().flatten().unsqueeze(0),
).item()
fused_cos = torch.nn.functional.cosine_similarity(
    tq_output.float().flatten().unsqueeze(0),
    tq_output_swz.float().flatten().unsqueeze(0),
).item()

print(f"Scores cosine (baseline vs swizzle):  {score_cos:.6f}")
print(f"Fused cosine  (baseline vs swizzle):  {fused_cos:.6f}")
print(f"Swizzle top-1: {tq_scores_swz.argmax().item()}")
print(f"Baseline top-1: {tq_scores.argmax().item()}")

## 8. Cross-check: cuTile vs PyTorch reference

In [ ]:
compressed_k_ref = engine.compress_keys_pytorch(K)
compressed_v_ref = engine.compress_values_pytorch(V)
scores_ref = engine.attention_scores_pytorch(Q, compressed_k_ref)
output_ref = engine.fused_attention_pytorch(Q, compressed_k_ref, compressed_v_ref)

k_idx_match = (compressed_k["indices"] == compressed_k_ref["indices"]).float().mean()
v_idx_match = (compressed_v["indices"] == compressed_v_ref["indices"]).float().mean()
score_diff = (tq_scores - scores_ref).abs().mean()
fused_cos = torch.nn.functional.cosine_similarity(
    tq_output.float().flatten().unsqueeze(0),
    output_ref.float().flatten().unsqueeze(0),
).item()

print(f"Key index match   (cuTile vs PyTorch): {k_idx_match:.4f}")
print(f"Value index match (cuTile vs PyTorch): {v_idx_match:.4f}")
print(f"Score mean abs diff:                    {score_diff:.6f}")
print(f"Fused output cosine sim:                {fused_cos:.4f}")

## 9. Latency benchmark (baseline vs swizzle)

In [ ]:
def bench(fn, warmup=20, runs=100):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs * 1e6

header = f"{'seq_k':>8} | {'FP16 matmul':>14} | {'fused':>10} | {'fused+swz':>12}"
print(header)
print("-" * len(header))

for seq_k in [512, 1024, 2048, 4096, 8192, 16384]:
    q = torch.randn(1, 128, device="cuda", dtype=torch.float16)
    k = torch.randn(seq_k, 128, device="cuda", dtype=torch.float16)
    v = torch.randn(seq_k, 128, device="cuda", dtype=torch.float16)

    ck = engine.launch_compress_keys(k)
    cv = engine.launch_compress_values(v)

    fp16_us = bench(lambda: q.float() @ k.float().T)
    ct_fused_us = bench(lambda: engine.launch_fused_attention(q, ck, cv))
    ct_fused_swz = bench(lambda: engine.launch_fused_attention(q, ck, cv, use_swizzle=True))

    print(f"{seq_k:>8} | {fp16_us:>11.1f} us | {ct_fused_us:>7.1f} us | {ct_fused_swz:>9.1f} us")

## 10. Needle-in-haystack retrieval

In [ ]:
for seq_k in [512, 2048, 8192]:
    keys = torch.randn(seq_k, 128, device="cuda")
    keys = keys / keys.norm(dim=-1, keepdim=True)

    needle = seq_k // 3
    query = keys[needle].unsqueeze(0)

    true_top1 = (query @ keys.T).argmax().item()

    ck = engine.launch_compress_keys(keys.half())
    sc = engine.launch_attention_scores(query.half(), ck)
    tq_top1 = sc.argmax().item()
    top5 = sc.topk(5).indices.tolist()[0]

    tag = "EXACT" if tq_top1 == true_top1 else ("TOP-5" if true_top1 in top5 else "MISS")
    print(f"seq={seq_k:>5d}: needle={needle}, true={true_top1}, tq={tq_top1} [{tag}]")

## 11. Unbiasedness of QJL estimator

In [ ]:
n = 2000
x = torch.randn(n, 128, device="cuda")
x = x / x.norm(dim=-1, keepdim=True)
y = torch.randn(n, 128, device="cuda")
y = y / y.norm(dim=-1, keepdim=True)

true_ip = (x * y).sum(dim=-1)

ck = engine.launch_compress_keys(x.half())
sc = engine.launch_attention_scores(y.half(), ck)
est_ip = torch.diag(sc) / engine.scale

bias = (est_ip - true_ip).mean()
rmse = ((est_ip - true_ip) ** 2).mean().sqrt()
corr = torch.corrcoef(torch.stack([true_ip, est_ip]))[0, 1]

print(f"Mean bias:   {bias:.6f}  (should be ~0)")
print(f"RMSE:        {rmse:.6f}")
print(f"Correlation: {corr:.4f}")

## 12. Compression ratios

In [ ]:
for bits in [2, 3, 4]:
    eng_tmp = TurboQuantEngine(head_dim=128, total_bits=bits, device="cpu")
    for ctx in [2048, 4096, 8192, 16384]:
        s = eng_tmp.compressed_size_bytes(ctx)
        print(f"  {bits}-bit @ {ctx:>5d} tokens: "
              f"{s['total_bytes']/1024:.0f} KB vs {s['fp16_bytes']/1024:.0f} KB FP16  "
              f"({s['compression_ratio']:.2f}x)")
    print()

## 13. B200 Roofline Analysis

Why does TurboQuant help even though it does *more* compute?
Both standard and compressed attention are **memory-bandwidth-bound** on Blackwell.
The kernel that reads less data wins.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# B200 Blackwell specifications
B200_FP16_TC_TFLOPS = 2250     # FP16 Tensor Core peak (dense)
B200_HBM_BW_TBS     = 8.0     # HBM3e bandwidth in TB/s
B200_HBM_GB         = 192     # HBM3e capacity

B200_PEAK = B200_FP16_TC_TFLOPS * 1e12   # FLOP/s
B200_BW   = B200_HBM_BW_TBS * 1e12       # bytes/s
RIDGE     = B200_PEAK / B200_BW           # ~281 FLOP/byte

d = HEAD_DIM  # 128

def ai_fp16(N):
    """Standard FP16 decode attention: Q@K^T + softmax + P@V"""
    flops = 4*N*d + 5*N
    mem   = 4*N*d + 4*d
    return flops / mem

def ai_tq_current(N):
    """Current TQ impl: loads pre-reconstructed K_mse (FP16), Signs, R_norms, V (FP16)"""
    flops = 6*N*d + 7*N        # two extra mma (QJL term) + correction
    mem   = 5*N*d + 2*N + 4*d  # K_mse + Signs + R_norms + V + Q + Q_proj
    return flops / mem

def ai_tq_fused(N, mse_bits=2, val_bits=3):
    """Fully-fused TQ: loads compressed data, reconstructs on-chip via Tensor Core mma"""
    recon_flops = 2 * 2*N*d*d  # un-rotate K and V: (N,d)@(d,d) each
    attn_flops  = 6*N*d + 7*N  # same as current (attention + QJL)
    flops = recon_flops + attn_flops

    mem = (N*d*mse_bits/8       # key indices
         + N*d/8                # QJL signs (1-bit)
         + 4*N                  # key norms + residual norms
         + N*d*val_bits/8       # value indices
         + 2*N                  # value norms
         + 4*d)                 # Q, Q_proj, output
    return flops / mem

# build roofline
ai_axis = np.logspace(-0.5, 3.5, 600)
roof    = np.minimum(B200_PEAK, B200_BW * ai_axis) / 1e12

fig, ax = plt.subplots(figsize=(11, 6))
ax.loglog(ai_axis, roof, "k-", linewidth=2.5, label="B200 roofline")
ax.axvline(RIDGE, color="gray", ls=":", alpha=0.5)
ax.text(RIDGE*1.15, B200_FP16_TC_TFLOPS*0.82, f"ridge = {RIDGE:.0f}", fontsize=9, color="gray")

seq_ks = [512, 2048, 8192]
colors_fp16 = "#2196F3"
colors_tq   = "#F44336"
colors_fused = "#4CAF50"

for N in seq_ks:
    a1 = ai_fp16(N);        t1 = min(B200_PEAK, B200_BW*a1)/1e12
    a2 = ai_tq_current(N);  t2 = min(B200_PEAK, B200_BW*a2)/1e12
    a3 = ai_tq_fused(N);    t3 = min(B200_PEAK, B200_BW*a3)/1e12

    ms = 10 if N == 2048 else 6
    al = 1.0 if N == 2048 else 0.45

    ax.plot(a1, t1, "o", color=colors_fp16,  ms=ms, alpha=al)
    ax.plot(a2, t2, "s", color=colors_tq,    ms=ms, alpha=al)
    ax.plot(a3, t3, "^", color=colors_fused,  ms=ms, alpha=al)

    if N == 2048:
        ax.annotate(f"FP16 (N={N})",     (a1, t1), xytext=(8,-12), textcoords="offset points", fontsize=8)
        ax.annotate(f"TQ current",       (a2, t2), xytext=(8, 6),  textcoords="offset points", fontsize=8)
        ax.annotate(f"TQ fused (target)", (a3, t3), xytext=(8,-12), textcoords="offset points", fontsize=8)

ax.plot([], [], "o", color=colors_fp16,  label="FP16 standard attention")
ax.plot([], [], "s", color=colors_tq,    label="TurboQuant (current impl)")
ax.plot([], [], "^", color=colors_fused, label="TurboQuant (fully fused)")

ax.fill_between([0.3, RIDGE], 0.1, 5000, alpha=0.04, color="blue",  label="_mem-bound zone")
ax.fill_between([RIDGE, 3000], 0.1, 5000, alpha=0.04, color="green", label="_compute-bound zone")
ax.text(1.5, 3500, "MEMORY-BOUND", fontsize=10, color="#1565C0", alpha=0.6)
ax.text(400, 3500, "COMPUTE-BOUND", fontsize=10, color="#2E7D32", alpha=0.6)

ax.set_xlabel("Arithmetic Intensity (FLOP/byte)", fontsize=12)
ax.set_ylabel("Attainable Performance (TFLOP/s)", fontsize=12)
ax.set_title("B200 Roofline: TurboQuant Compressed Attention", fontsize=14)
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(0.3, 3000)
ax.set_ylim(0.5, 5000)
ax.grid(True, alpha=0.25, which="both")
plt.tight_layout()
plt.savefig("roofline_b200.png", dpi=150)
plt.show()

print(f"\nB200 Blackwell: {B200_FP16_TC_TFLOPS} TFLOP/s FP16 TC, {B200_HBM_BW_TBS} TB/s HBM3e, {B200_HBM_GB} GB")
print(f"Ridge point: {RIDGE:.0f} FLOP/byte\n")

for N in seq_ks:
    a1, a2, a3 = ai_fp16(N), ai_tq_current(N), ai_tq_fused(N)
    print(f"seq_k={N:>5d}:  FP16 AI={a1:.2f}  TQ_current AI={a2:.2f}  TQ_fused AI={a3:.1f}")

print(f"\nAll points sit at AI << {RIDGE:.0f} (memory-bound region)")
print("Current TQ reads MORE data (K_mse + Signs + R_norms + V) → slower on micro-benchmarks")
print("Fully-fused TQ shifts to compute-bound by loading only compressed data and reconstructing on-chip")
print("Storage savings (5x) still hold regardless → 5x longer context at same memory budget")

## 14. End-to-End: Compressed KV Cache on a Real Transformer

Load a transformer LLM, extract its KV cache after prefill, compress it with our cuTile kernels,
and verify attention quality is preserved on real model activations.
This follows the integration pattern used in [NVIDIA TileGym](https://github.com/NVIDIA/TileGym).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Qwen2-1.5B has head_dim=128, matching our kernels. Works without HF auth.
# Swap to Qwen/Qwen2-7B or meta-llama/Meta-Llama-3.1-8B for a bigger demo.
MODEL_ID = "Qwen/Qwen2-1.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="cuda",
)

print(f"Model:  {MODEL_ID}")
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
print(f"GPU memory after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### 14a. Prefill and extract KV cache

In [ ]:
prompt = (
    "Tile-based GPU programming represents a fundamental shift in how we think about "
    "parallel computation. Rather than operating on individual elements, tile-based "
    "approaches process structured blocks of data that align with the hardware's memory "
    "hierarchy and tensor core layout. This enables"
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model(**inputs, use_cache=True)
    past_kv = outputs.past_key_values

# DynamicCache (transformers >= 4.36) vs legacy tuple-of-tuples
if hasattr(past_kv, "key_cache"):
    kv_keys = past_kv.key_cache
    kv_vals = past_kv.value_cache
else:
    kv_keys = [kv[0] for kv in past_kv]
    kv_vals = [kv[1] for kv in past_kv]

num_layers = len(kv_keys)
num_heads  = kv_keys[0].shape[1]
seq_len    = kv_keys[0].shape[2]
head_dim_m = kv_keys[0].shape[3]

uncompressed_bytes = sum(k.nbytes + v.nbytes for k, v in zip(kv_keys, kv_vals))

print(f"KV cache extracted:")
print(f"  Layers:    {num_layers}")
print(f"  KV heads:  {num_heads}")
print(f"  Seq len:   {seq_len}")
print(f"  Head dim:  {head_dim_m}")
print(f"  Uncompressed size: {uncompressed_bytes / 1024:.1f} KB")

assert head_dim_m == HEAD_DIM, f"Model head_dim={head_dim_m} but kernels expect {HEAD_DIM}"

### 14b. Compress full KV cache with cuTile kernels

In [ ]:
tq_engine = TurboQuantEngine(
    head_dim=head_dim_m, total_bits=TOTAL_BITS, seed=SEED, device="cuda",
)

compressed_cache = []
t0 = time.perf_counter()

for layer_idx in range(num_layers):
    layer_ck, layer_cv = [], []
    for h in range(num_heads):
        k_h = kv_keys[layer_idx][0, h].half().contiguous()
        v_h = kv_vals[layer_idx][0, h].half().contiguous()
        layer_ck.append(tq_engine.launch_compress_keys(k_h))
        layer_cv.append(tq_engine.launch_compress_values(v_h))
    compressed_cache.append((layer_ck, layer_cv))

torch.cuda.synchronize()
compress_ms = (time.perf_counter() - t0) * 1000

sizes = tq_engine.compressed_size_bytes(seq_len)
compressed_total = sizes["total_bytes"] * num_heads * num_layers

print(f"Compressed {num_layers} layers x {num_heads} heads in {compress_ms:.1f} ms")
print(f"\nKV cache memory:")
print(f"  FP16 uncompressed:  {uncompressed_bytes / 1024:.1f} KB")
print(f"  TurboQuant {TOTAL_BITS}-bit:  {compressed_total / 1024:.1f} KB")
print(f"  Compression ratio:  {sizes['compression_ratio']:.2f}x")

### 14c. Attention quality on real model activations

In [ ]:
print("Per-layer attention quality (cosine sim vs FP16 reference):\n")

cos_scores = []
sample_layers = sorted(set([0, num_layers // 4, num_layers // 2, num_layers - 1]))

for layer_idx in sample_layers:
    for h in [0]:
        k_h = kv_keys[layer_idx][0, h].half()
        v_h = kv_vals[layer_idx][0, h].half()

        q_probe = torch.randn(1, head_dim_m, device="cuda", dtype=torch.float16)

        ck_h = compressed_cache[layer_idx][0][h]
        cv_h = compressed_cache[layer_idx][1][h]

        true_scores_h = (q_probe.float() @ k_h.float().T) * tq_engine.scale
        true_weights_h = torch.softmax(true_scores_h, dim=-1)
        true_out_h = (true_weights_h @ v_h.float()).half()

        tq_out_h = tq_engine.launch_fused_attention(q_probe, ck_h, cv_h)

        cos_h = torch.nn.functional.cosine_similarity(
            true_out_h.float().flatten().unsqueeze(0),
            tq_out_h.float().flatten().unsqueeze(0),
        ).item()
        cos_scores.append(cos_h)

        print(f"  Layer {layer_idx:>2d}, Head {h}: cosine = {cos_h:.4f}")

print(f"\n  Mean cosine across sampled layers: {sum(cos_scores)/len(cos_scores):.4f}")

### 14d. Memory scaling: TurboQuant vs FP16 at long context

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ctx_lengths = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
fp16_mb  = []
tq_mb    = []
ratios   = []

for ctx in ctx_lengths:
    s = tq_engine.compressed_size_bytes(ctx)
    fp16_total = s["fp16_bytes"]   * num_heads * num_layers
    tq_total   = s["total_bytes"]  * num_heads * num_layers
    fp16_mb.append(fp16_total / 1e6)
    tq_mb.append(tq_total / 1e6)
    ratios.append(s["compression_ratio"])

ax1.plot(ctx_lengths, fp16_mb, "o-", color="#2196F3", label="FP16 KV cache", linewidth=2)
ax1.plot(ctx_lengths, tq_mb,   "s-", color="#4CAF50", label=f"TurboQuant {TOTAL_BITS}-bit", linewidth=2)

ax1.axhline(B200_HBM_GB * 1e3 * 0.5, color="red", ls="--", alpha=0.4,
            label=f"50% of B200 HBM ({B200_HBM_GB//2} GB)")
ax1.set_xlabel("Context Length (tokens)")
ax1.set_ylabel("KV Cache Size (MB)")
ax1.set_title(f"KV Cache Memory ({num_layers}L x {num_heads}H x {head_dim_m}d)")
ax1.set_xscale("log", base=2)
ax1.set_yscale("log")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.25, which="both")

ax2.bar(range(len(ctx_lengths)), ratios, color="#FF9800")
ax2.set_xticks(range(len(ctx_lengths)))
ax2.set_xticklabels([f"{c//1024}K" for c in ctx_lengths], fontsize=8)
ax2.set_xlabel("Context Length")
ax2.set_ylabel("Compression Ratio (x)")
ax2.set_title(f"TurboQuant {TOTAL_BITS}-bit Compression Ratio")
ax2.axhline(ratios[0], color="gray", ls=":", alpha=0.5)
ax2.set_ylim(0, max(ratios) * 1.2)
ax2.grid(True, alpha=0.25, axis="y")

plt.tight_layout()
plt.savefig("memory_scaling_b200.png", dpi=150)
plt.show()

for ctx, fp, tq, r in zip(ctx_lengths, fp16_mb, tq_mb, ratios):
    print(f"  {ctx:>6d} tokens: {fp:>8.1f} MB FP16 → {tq:>8.1f} MB TQ ({r:.2f}x)")

### 14e. Greedy decode: standard vs TurboQuant attention (single layer)

In [ ]:
# Standard generation (unmodified model)
with torch.no_grad():
    gen_standard = model.generate(
        **inputs, max_new_tokens=60, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
text_standard = tokenizer.decode(gen_standard[0], skip_special_tokens=True)

print("=== Standard FP16 generation ===")
print(text_standard)
print(f"\nTokens generated: {gen_standard.shape[1] - inputs['input_ids'].shape[1]}")

### 14f. Generate with decompressed value cache

Build a `DynamicCache` using original FP16 keys + decompressed TurboQuant values,
then run greedy decoding. Keys stay original because the QJL correction only works
inside the dot product (that's what the fused attention kernel in 14g handles).
This isolates value compression quality.

In [ ]:
from transformers import DynamicCache
from IPython.display import clear_output

recon_cache = DynamicCache()
for layer_idx in range(num_layers):
    v_heads = []
    for h in range(num_heads):
        cv_h = compressed_cache[layer_idx][1][h]
        v_heads.append(tq_engine.decompress_values_pytorch(cv_h))
    k_layer = kv_keys[layer_idx]
    v_layer = torch.stack(v_heads, dim=0).unsqueeze(0)
    recon_cache.update(k_layer, v_layer, layer_idx)

MAX_NEW_TOKENS = 60
generated_ids = []

sizes = tq_engine.compressed_size_bytes(seq_len)
compressed_bytes = sizes["total_bytes"] * num_heads * num_layers
fp16_bytes = sizes["fp16_bytes"] * num_heads * num_layers

next_token = outputs.logits[:, -1:].argmax(dim=-1)
generated_ids.append(next_token.item())

t0 = time.perf_counter()
with torch.no_grad():
    for step in range(1, MAX_NEW_TOKENS):
        position_ids = torch.tensor([[seq_len + step - 1]], device="cuda")
        out = model(
            input_ids=next_token,
            past_key_values=recon_cache,
            position_ids=position_ids,
            use_cache=True,
        )
        recon_cache = out.past_key_values
        next_token = out.logits[:, -1:].argmax(dim=-1)
        generated_ids.append(next_token.item())
        if next_token.item() == tokenizer.eos_token_id:
            break

gen_time = time.perf_counter() - t0
text_reconstructed = tokenizer.decode(generated_ids, skip_special_tokens=True)
total_tokens = len(generated_ids)
real_tps = total_tokens / gen_time

# --- Dramatic reveal for screen recording ---
CHUNK_SIZE = 2
REVEAL_DELAY = 0.30

words = text_reconstructed.split()
num_chunks = (len(words) + CHUNK_SIZE - 1) // CHUNK_SIZE

for c in range(1, num_chunks + 1):
    shown_words = words[:c * CHUNK_SIZE]
    frac = min(c / num_chunks, 1.0)
    tok_est = int(frac * total_tokens)

    cur_sizes = tq_engine.compressed_size_bytes(seq_len + tok_est)
    cur_comp = cur_sizes["total_bytes"] * num_heads * num_layers
    cur_fp16 = cur_sizes["fp16_bytes"] * num_heads * num_layers

    bar_len = 30
    filled = int(frac * bar_len)
    bar = "\u2588" * filled + "\u2591" * (bar_len - filled)

    import textwrap
    P = "          "
    W = 64

    def wrap_text(txt):
        lines = textwrap.wrap(txt, width=W)
        return "\n".join(P + line for line in lines)

    clear_output(wait=True)
    print("\n" * 6)
    print(f"{P}TurboQuant \u2014 Generating from Compressed KV Cache")
    print(f"{P}{'='*W}")
    print(f"{P}Tokens   {tok_est:>3d}/{total_tokens}  [{bar}]")
    print(f"{P}Cache    {num_layers} layers \u00d7 {num_heads} KV heads \u00d7 {head_dim_m}d")
    print(f"{P}TQ size  {cur_comp/1024:>7.1f} KB    FP16  {cur_fp16/1024:>7.1f} KB")
    print(f"{P}Ratio    {cur_fp16/cur_comp:.2f}x              {real_tps:.1f} tok/s")
    print(f"{P}{'='*W}")
    print()
    print(wrap_text(" ".join(shown_words)))
    print("\n" * 6)

    time.sleep(REVEAL_DELAY)

import textwrap
P = "          "
W = 64

def wrap_text(txt):
    lines = textwrap.wrap(txt, width=W)
    return "\n".join(P + line for line in lines)

clear_output(wait=True)
print("\n" * 6)
print(f"{P}TurboQuant \u2014 Generation Complete")
print(f"{P}{'='*W}")
print(f"{P}Tokens   {total_tokens:>3d}/{total_tokens}  [{'\u2588' * bar_len}]")
print(f"{P}Cache    {num_layers} layers \u00d7 {num_heads} KV heads \u00d7 {head_dim_m}d")
print(f"{P}TQ size  {compressed_bytes/1024:>7.1f} KB    FP16  {fp16_bytes/1024:>7.1f} KB")
print(f"{P}Ratio    {fp16_bytes/compressed_bytes:.2f}x              {real_tps:.1f} tok/s")
print(f"{P}{'='*W}")
print()
print(wrap_text(text_reconstructed))
print()
print(f"{P}{total_tokens} tokens in {gen_time:.2f}s")
print("\n" * 6)

### 14g. TurboQuant attention quality — comprehensive analysis

Detailed comparison of TurboQuant fused attention vs standard FP16 attention across
multiple layers, all KV heads, and diverse query vectors (random probes + actual key
vectors from the cache). Measures output cosine similarity, attention weight fidelity,
and score error.

In [ ]:
import torch.nn.functional as F

sample_layers = sorted(set([0, num_layers // 4, num_layers // 2, 3 * num_layers // 4, num_layers - 1]))

results = []

for layer_idx in sample_layers:
    for h in range(num_heads):
        k_h = kv_keys[layer_idx][0, h].half()
        v_h = kv_vals[layer_idx][0, h].half()
        ck_h = compressed_cache[layer_idx][0][h]
        cv_h = compressed_cache[layer_idx][1][h]

        probes = [torch.randn(1, head_dim_m, device="cuda", dtype=torch.float16) for _ in range(4)]
        key_indices = [0, seq_len // 4, seq_len // 2, seq_len - 1]
        probes += [k_h[idx:idx+1] for idx in key_indices]

        for q_probe in probes:
            true_scores = (q_probe.float() @ k_h.float().T) * tq_engine.scale
            true_weights = torch.softmax(true_scores, dim=-1)
            true_out = (true_weights @ v_h.float()).half()

            tq_out = tq_engine.launch_fused_attention(q_probe, ck_h, cv_h)

            out_cos = F.cosine_similarity(
                true_out.float().flatten().unsqueeze(0),
                tq_out.float().flatten().unsqueeze(0),
            ).item()

            tq_scores = tq_engine.attention_scores_pytorch(q_probe, ck_h)
            tq_weights = torch.softmax(tq_scores.float(), dim=-1)

            weight_cos = F.cosine_similarity(
                true_weights.flatten().unsqueeze(0),
                tq_weights.flatten().unsqueeze(0),
            ).item()

            max_weight_err = (true_weights - tq_weights).abs().max().item()

            results.append({
                "layer": layer_idx, "head": h,
                "out_cos": out_cos, "weight_cos": weight_cos,
                "max_weight_err": max_weight_err,
            })

out_cos_vals = [r["out_cos"] for r in results]
wt_cos_vals  = [r["weight_cos"] for r in results]
max_err_vals = [r["max_weight_err"] for r in results]
n = len(results)

print(f"=== TurboQuant Attention Quality ===")
print(f"Tested: {n} queries ({len(sample_layers)} layers × {num_heads} heads × 8 probes)\n")

print(f"Attention OUTPUT cosine similarity:")
print(f"  Mean: {sum(out_cos_vals)/n:.6f}   Min: {min(out_cos_vals):.6f}   Max: {max(out_cos_vals):.6f}")

print(f"\nAttention WEIGHT cosine similarity:")
print(f"  Mean: {sum(wt_cos_vals)/n:.6f}   Min: {min(wt_cos_vals):.6f}   Max: {max(wt_cos_vals):.6f}")

print(f"\nMax absolute weight error per query:")
print(f"  Mean: {sum(max_err_vals)/n:.6f}   Max: {max(max_err_vals):.6f}")

print(f"\n--- Per-layer breakdown (mean output cosine) ---")
for l in sample_layers:
    layer_cos = [r["out_cos"] for r in results if r["layer"] == l]
    print(f"  Layer {l:>2d}:  {sum(layer_cos)/len(layer_cos):.6f}  (n={len(layer_cos)})")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(out_cos_vals, bins=30, color="#76B900", edgecolor="white", alpha=0.9)
axes[0].set_title("Output Cosine Similarity")
axes[0].set_xlabel("cosine(FP16, TQ)")
axes[0].axvline(sum(out_cos_vals)/n, color="red", linestyle="--", label=f"mean={sum(out_cos_vals)/n:.4f}")
axes[0].legend()

axes[1].hist(wt_cos_vals, bins=30, color="#76B900", edgecolor="white", alpha=0.9)
axes[1].set_title("Weight Cosine Similarity")
axes[1].set_xlabel("cosine(FP16 weights, TQ weights)")
axes[1].axvline(sum(wt_cos_vals)/n, color="red", linestyle="--", label=f"mean={sum(wt_cos_vals)/n:.4f}")
axes[1].legend()

axes[2].hist(max_err_vals, bins=30, color="#E04040", edgecolor="white", alpha=0.9)
axes[2].set_title("Max Absolute Weight Error")
axes[2].set_xlabel("max |w_fp16 - w_tq|")
axes[2].axvline(max(max_err_vals), color="red", linestyle="--", label=f"worst={max(max_err_vals):.4f}")
axes[2].legend()

plt.suptitle(f"TurboQuant vs FP16 Attention — {n} queries across {len(sample_layers)} layers", fontsize=13)
plt.tight_layout()
plt.show()

### 14h. Side-by-side: Standard FP16 vs Decompressed Values

Compare standard generation against generation from the decompressed TurboQuant
value cache (with original FP16 keys). Token-by-token agreement shows the
compression round-trip preserves generation quality.

In [ ]:
std_tokens = tokenizer.encode(text_standard, add_special_tokens=False)
recon_tokens = tokenizer.encode(text_reconstructed, add_special_tokens=False) if generated_ids else []

min_len = min(len(std_tokens), len(recon_tokens)) if recon_tokens else 0

if min_len > 0:
    match_recon = sum(a == b for a, b in zip(std_tokens[:min_len], recon_tokens[:min_len]))
    print(f"Token agreement (first {min_len} tokens):\n")
    print(f"  Standard vs Decompressed V:  {match_recon}/{min_len} ({100*match_recon/min_len:.1f}%)")

print(f"\n{'='*70}")
print(f"{'Standard FP16':^70}")
print(f"{'='*70}")
print(text_standard)

print(f"\n{'='*70}")
print(f"{'Decompressed Values (orig keys + TQ values)':^70}")
print(f"{'='*70}")
print(prompt + text_reconstructed)

if min_len > 0:
    for i in range(min_len):
        if std_tokens[i] != recon_tokens[i]:
            ctx_start = max(0, i - 3)
            ctx = tokenizer.decode(std_tokens[ctx_start:i])
            std_word = tokenizer.decode([std_tokens[i]])
            recon_word = tokenizer.decode([recon_tokens[i]])
            print(f"\nFirst divergence at token {i}: ...{ctx}[{std_word!r} vs {recon_word!r}]")
            break
    else:
        print(f"\nAll {min_len} tokens match exactly.")

### 14i. Memory footprint breakdown

In [ ]:
sizes = tq_engine.compressed_size_bytes(seq_len)
n = num_heads * num_layers

key_bytes = sizes["key_bytes"] * n
val_bytes = sizes["val_bytes"] * n
tq_total  = sizes["total_bytes"] * n
fp16_k    = seq_len * head_dim_m * 2 * n
fp16_v    = seq_len * head_dim_m * 2 * n
fp16_total = fp16_k + fp16_v

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

categories = ["FP16\n(uncompressed)", f"TurboQuant\n({TOTAL_BITS}-bit)"]
bar_k = [fp16_k / 1024, key_bytes / 1024]
bar_v = [fp16_v / 1024, val_bytes / 1024]

x = range(len(categories))
ax1.bar(x, bar_k, label="Keys", color="#2196F3", width=0.5)
ax1.bar(x, bar_v, bottom=bar_k, label="Values", color="#4CAF50", width=0.5)

for i, (k, v) in enumerate(zip(bar_k, bar_v)):
    ax1.text(i, k + v + 1, f"{k+v:.1f} KB", ha="center", fontsize=10, fontweight="bold")

ax1.set_xticks(x)
ax1.set_xticklabels(categories)
ax1.set_ylabel("KV Cache Size (KB)")
ax1.set_title(f"Memory: {num_layers}L x {num_heads}H x {seq_len} tokens")
ax1.legend()
ax1.grid(True, alpha=0.2, axis="y")

d = head_dim_m
mse_bits = tq_engine.mse_bits
key_mse    = seq_len * d * mse_bits / 8 * n / 1024
key_qjl    = seq_len * d * 1 / 8 * n / 1024
key_norms  = seq_len * 2 * 2 * n / 1024
val_mse    = seq_len * d * TOTAL_BITS / 8 * n / 1024
val_norms  = seq_len * 2 * n / 1024

components = ["Key\nMSE idx", "Key\nQJL signs", "Key\nnorms", "Value\nMSE idx", "Value\nnorms"]
comp_sizes = [key_mse, key_qjl, key_norms, val_mse, val_norms]
colors = ["#2196F3", "#64B5F6", "#90CAF9", "#4CAF50", "#81C784"]

ax2.bar(range(len(components)), comp_sizes, color=colors, width=0.6)
for i, s in enumerate(comp_sizes):
    ax2.text(i, s + 0.2, f"{s:.1f}", ha="center", fontsize=9)

ax2.set_xticks(range(len(components)))
ax2.set_xticklabels(components, fontsize=8)
ax2.set_ylabel("Size (KB)")
ax2.set_title(f"TurboQuant {TOTAL_BITS}-bit Component Breakdown")
ax2.grid(True, alpha=0.2, axis="y")

plt.tight_layout()
plt.savefig("memory_breakdown_b200.png", dpi=150)
plt.show()

print(f"\nCompression ratio: {fp16_total/tq_total:.2f}x  ({fp16_total/1024:.1f} KB -> {tq_total/1024:.1f} KB)")

### 14j. Long context stress test: where does FP16 run out of memory?

In [ ]:
model_params_gb = sum(p.nbytes for p in model.parameters()) / 1e9
avail_for_kv = B200_HBM_GB - model_params_gb - 2

context_lengths = [4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576]

print(f"Model weights: {model_params_gb:.1f} GB | Available for KV: {avail_for_kv:.1f} GB")
print(f"Architecture:  {num_layers}L x {num_heads} KV heads x {head_dim_m}d\n")
print(f"{'Context':>10s}  {'FP16 KV':>10s}  {'TQ '+str(TOTAL_BITS)+'-bit':>10s}  {'Ratio':>7s}  {'FP16':>12s}  {'TQ':>12s}")
print("-" * 70)

fp16_oom_ctx = None
tq_oom_ctx = None

for ctx in context_lengths:
    s = tq_engine.compressed_size_bytes(ctx)
    fp16_gb = s["fp16_bytes"] * num_heads * num_layers / 1e9
    tq_gb = s["total_bytes"] * num_heads * num_layers / 1e9

    fp16_status = "OK" if fp16_gb < avail_for_kv else "OOM"
    tq_status = "OK" if tq_gb < avail_for_kv else "OOM"

    if fp16_status == "OOM" and fp16_oom_ctx is None:
        fp16_oom_ctx = ctx
    if tq_status == "OOM" and tq_oom_ctx is None:
        tq_oom_ctx = ctx

    print(f"{ctx:>10,d}  {fp16_gb:>9.2f}G  {tq_gb:>9.2f}G  {s['compression_ratio']:>6.2f}x"
          f"  {'  '+fp16_status:>12s}  {'  '+tq_status:>12s}")

print(f"\n{'='*70}")
if fp16_oom_ctx:
    print(f"FP16 hits OOM at {fp16_oom_ctx:,} tokens")
else:
    print(f"FP16 fits all tested context lengths")

if tq_oom_ctx:
    print(f"TurboQuant {TOTAL_BITS}-bit hits OOM at {tq_oom_ctx:,} tokens")
else:
    print(f"TurboQuant {TOTAL_BITS}-bit fits all tested context lengths (up to {context_lengths[-1]:,})")

if fp16_oom_ctx and not tq_oom_ctx:
    print(f"\nTurboQuant extends max context by {context_lengths[-1]//fp16_oom_ctx}x+ before running out of memory")

## 15. Summary

| | |
|---|---|
| **Kernels** | Key compression, value compression, attention scores, fused attention, value decompression — all cuTile |
| **Optimizations** | Block swizzling, `exp2` flush-to-zero, approximate division, pipelined loads (toggleable) |
| **Roofline** | Current impl is memory-bound (AI ≈ 1.2); fully-fused target is compute-bound (AI ≈ 650) |
| **Storage** | ~5x compression on KV cache → proportionally longer context at fixed memory |
| **Quality** | Unbiased attention estimation (QJL), high cosine similarity on real model activations |
| **Hardware** | Targets Blackwell B200: 2,250 TFLOP/s FP16 TC, 8 TB/s HBM3e, 192 GB |